In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import LocalOutlierFactor
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from enum import Enum
import os
from sklearn.ensemble import GradientBoostingClassifier
import time
from datetime import datetime
import torch
from typing import List, Dict, Any

In [5]:
from temain import TemainProcess
import importlib
import v2_utils
import generators
import autoencoders
importlib.reload(v2_utils)
# importlib.reload(generators)
# importlib.reload(autoencoders)
from v2_utils import CSVDataLoader, ExperimentRunner, plot_with_anomalies, AdvancedDetectionEvaluator, BaseAnomalyDetector, ExperimentRunner
from autoencoders import RecurrentAutoencoder
from generators import ParallelDataGenerator, SingleProcessGenerator, N_DV

In [6]:
def run_experiments_by_anomaly(
        anomaly_numbers: List[int],
        data_dir_template: str = "./gen_data/gen_data_{}",
        model_params_list=None,
        file_fraction=0.01,
        anomaly_ratio=0.11,
        max_samples=None,
        result_title="",
        results_path="./results/my",
        window=30,
        autoencoder=None,
    ):
    timestamp = datetime.now().strftime("%Y-%m-%d_%H:%M:%S")
    # timestamp = "-"
    normal_dir = data_dir_template.format(0)
    all_best_models = []
    all_best_per_detector = []
    
    for i in anomaly_numbers:
        print(f"\n{'='*50}")
        print(f"Запуск экспериментов для аномалии {i}")
        print(f"{'='*50}")
        start_time = time.time()
        
        data_dir = data_dir_template.format(i)
        
        
        data_gen = CSVDataLoader(data_dir, 
                                 file_fraction=file_fraction,
                                 normal_dir=normal_dir, 
                                 max_samples=max_samples,
                                 anomaly_ratio=anomaly_ratio,
                                 shuffle=True,
                                 normalize=True)
        
        runner = ExperimentRunner(data_gen, evaluator=AdvancedDetectionEvaluator)
        
        runner.register_detector(
            'IsolationForest', 
            IsolationForest,
            {'random_state': 42, 'contamination': 0.1}
        )

        runner.register_detector(
            'LocalOutlierFactor', 
            LocalOutlierFactor,
            {'contamination': 0.1, "novelty": True}
        )

        runner.register_detector(
            'OneClassSVM', 
            OneClassSVM,
            {'kernel': 'linear', 'nu': 0.1} # {'kernel': 'rbf', 'gamma': 'auto', 'nu': 0.1}
        )
        # runner.register_detector(
        #     'GradientBoosting', 
        #     GradientBoostingClassifier,
        #     {'random_state': 42}
        # )
        
        results = runner.run_comprehensive_experiments(
            model_params_list=model_params_list,
            test_delays=None,
            autoencoder=autoencoder,
            window=window,
        )
        
        filename = f"{results_path}/experiments_results_{i}.csv"
        # results.to_csv(filename, index=False)
        # print(f"Результаты сохранены в {filename}")
        
        best_pr = runner.get_best_models('pr_auc', top_k=1)
        best_pr['anomaly_number'] = i
        best_pr['best_by_metric'] = 'pr_auc'
        all_best_models.append(best_pr)
        scores_filename = f"{results_path}/scores_{timestamp}_ANOM{i}_{result_title}.npy"
        np.save(scores_filename, np.stack([best_pr["test_scores"], best_pr["test_target"]]))
        print(f"Скоры и метки для test сохранены в {scores_filename}")
        
        best_roc = runner.get_best_models('roc_auc', top_k=1)
        best_roc['anomaly_number'] = i
        best_roc['best_by_metric'] = 'roc_auc'
        all_best_models.append(best_roc)
        # f1 ?
        
        best_per_detector = runner.get_best_per_detector('pr_auc')
        if not best_per_detector.empty:
            best_per_detector['anomaly_number'] = i
            all_best_per_detector.append(best_per_detector)
        if best_pr.iloc[0]["survival_km"]:
            best_pr.iloc[0]["survival_km"].plot_survival_function()
            plt.title('Функции выживаемости времени обнаружения аномалий')
            plt.ylabel('Вероятность необнаружения аномалии')
            plt.xlabel('Время (шаги)')
            # plt.ylim(0, 1)
            plt.grid(True)
            plt.show()
        
        print(f"Аномалия {i}:")
        print(f"  - Лучший PR AUC: {best_pr['pr_auc'].iloc[0]:.4f} ({best_pr['detector'].iloc[0]})")
        print(f"  - Лучший ROC AUC: {best_roc['roc_auc'].iloc[0]:.4f} ({best_roc['detector'].iloc[0]})")
        # print(f"  - Лучший F1: {best_f1['f1_score'].iloc[0]:.4f} ({best_f1['detector'].iloc[0]})")
        cur = time.time()
        print(f"\n-------------------\nВремя выполнения: {cur - start_time:.4f} секунд\n")
    
    if all_best_models and all_best_per_detector:
        summary_df = pd.concat(all_best_models, ignore_index=True)
        numeric_cols = summary_df.select_dtypes(include=['number', 'bool', 'int64', 'float64', 'string']).columns.tolist()
        numeric_cols += [col for col in summary_df.columns if summary_df[col].dtype == 'object' and all(isinstance(x, str) for x in summary_df[col].dropna().head(10))]
        summary_df = summary_df[numeric_cols]
        summary_filename = f"{results_path}/best_models_summary_{timestamp}_{result_title}.csv"
        summary_df.to_csv(summary_filename, index=False)
        print(f"\nСводная таблица сохранена в {summary_filename}")
        
        per_detector_df = pd.concat(all_best_per_detector, ignore_index=True)
        pivot_df = per_detector_df.pivot_table(
            index='anomaly_number', 
            columns='detector', 
            values='pr_auc',
            aggfunc='first'
        )
        pivot_filename = f"{results_path}/pr_auc_pivot_{timestamp}_{result_title}.csv"
        pivot_df.to_csv(pivot_filename)
        print(f"Сводная таблица PR AUC по детекторам сохранена в {pivot_filename}")
        return summary_df, pivot_df
    return None, None



In [ ]:
model_params_list = {
    'IsolationForest': [
        # {'n_estimators': 200, 'contamination': 0.1, "random_state": 42},
        {'n_estimators': 100, 'max_samples': 'auto', 'contamination': 0.1, 'random_state': 42},
        # {'n_estimators': 200, 'max_samples': 0.8, 'contamination': 0.05, 'random_state': 42},
        # {'n_estimators': 50, 'max_samples': 1.0, 'contamination': 0.1, 'random_state': 42},
        # {'n_estimators': 100, 'max_samples': 0.5, 'contamination': 'auto', 'random_state': 42}
    ],
    'LocalOutlierFactor': [
        # {'n_neighbors': 50, 'contamination': 0.1, 'novelty': True},
        {'n_neighbors': 20, 'contamination': 0.1, 'novelty': True},
        # {'n_neighbors': 10, 'contamination': 0.05, 'novelty': True},
        # {'n_neighbors': 10, 'contamination': 0.1, 'novelty': True},
        # {'n_neighbors': 50, 'contamination': 0.15, 'novelty': True},
        # {'n_neighbors': 20, 'contamination': 'auto', 'novelty': True}
    ],
    'OneClassSVM': [
        # {'kernel': 'rbf', 'nu': 0.1},
        # {'kernel': 'rbf', 'nu': 0.15},
        {'kernel': 'linear', 'nu': 0.15},
        {'kernel': 'linear', 'nu': 0.2},
        {'kernel': 'linear', 'nu': 0.10},
        # {'kernel': 'poly', 'nu': 0.15},
    ],
    # 'GradientBoosting': [
    #     {'n_estimators': 100, 'max_depth': 3, 'learning_rate': 0.1, 'random_state': 42},
    #     {'n_estimators': 200, 'max_depth': 5, 'learning_rate': 0.05, 'random_state': 42},
    #     {'n_estimators': 100, 'max_depth': 5, 'learning_rate': 0.01, 'random_state': 42},
    # ]
}

autoencoder = RecurrentAutoencoder(
            input_dim=53,
            window_size=30,
            hidden_dim=64,
            latent_dim=32,
            num_layers=1
        )
model_saved_path = f"./models/new_autoencoder_w15.pth"
# model_saved_path = f"./models/new_autoencoder_layer2_w15.pth"

autoencoder.load_state_dict(torch.load(model_saved_path))
autoencoder.eval()

anomaly_numbers = list(range(1, 20))
# anomaly_numbers = [1]

summary, summary_detectors = run_experiments_by_anomaly(anomaly_numbers,
                                     data_dir_template="./gen_data_reduced/gen_data_{}",
                                     model_params_list=model_params_list,
                                     max_samples=100000,
                                     file_fraction=0.05,
                                     anomaly_ratio=0.08,
                                     window=15,
                                     result_title="new_reduced_data_unsupervised_with_autoencoder",
                                     results_path="results/cleared",
                                     autoencoder=autoencoder,
                                     )


Запуск экспериментов для аномалии 1


In [ ]:
summary

In [ ]:
summary_detectors

In [ ]:
# reduced, without windows
anomaly_numbers = list(range(1, 20))
# anomaly_numbers = [1]

summary, summary_detectors = run_experiments_by_anomaly(anomaly_numbers,
                                     data_dir_template="./gen_data_reduced/gen_data_{}",
                                     model_params_list=model_params_list,
                                     max_samples=100000,
                                     file_fraction=0.05,
                                     anomaly_ratio=0.08,
                                     window=None,
                                     result_title="new_reduced_data_unsupervised_no_windows",
                                     results_path="results/cleared",
                                     autoencoder=None
                                     )

In [ ]:
summary_detectors

In [ ]:
# not reduced, without windows
anomaly_numbers = list(range(1, 20))
# anomaly_numbers = [1]

summary, summary_detectors = run_experiments_by_anomaly(anomaly_numbers,
                                     data_dir_template="./gen_data_new/gen_data_{}",
                                     model_params_list=model_params_list,
                                     max_samples=100000,
                                     file_fraction=0.05,
                                     anomaly_ratio=0.08,
                                     window=None,
                                     result_title="new_data_unsupervised_no_windows",
                                     results_path="results/cleared",
                                     autoencoder=None
                                     )

In [ ]:
summary_detectors

In [ ]:
# not reduced, autoencoder
anomaly_numbers = list(range(1, 20))
# anomaly_numbers = [1]

summary, summary_detectors = run_experiments_by_anomaly(anomaly_numbers,
                                     data_dir_template="./gen_data_new/gen_data_{}",
                                     model_params_list=model_params_list,
                                     max_samples=100000,
                                     file_fraction=0.05,
                                     anomaly_ratio=0.08,
                                     window=None,
                                     result_title="new_data_unsupervisedwith_autoencoder",
                                     results_path="results/cleared",
                                     autoencoder=autoencoder
                                     )

In [ ]:
summary_detectors